# GPU worker trên Colab

Notebook chạy Celery worker của `ai_workers` trên GPU Colab và nhận job từ website qua queue `gpu`.
Website tự chuyển job sang worker CPU local khi notebook này tắt.

Trước khi chạy, thêm vào **Colab Secrets** (biểu tượng chìa khóa bên trái) và bật quyền truy cập notebook:

| Secret | Bắt buộc | Ghi chú |
|---|---|---|
| `SHARED_REDIS_URL` | có | Cùng Redis với backend, ví dụ `rediss://default:<pass>@<host>:6379?ssl_cert_reqs=CERT_REQUIRED` |
| `CF_R2_ACCOUNT_ID`, `CF_R2_ACCESS_KEY_ID`, `CF_R2_SECRET_ACCESS_KEY`, `CF_R2_BUCKET_NAME`, `CF_R2_PUBLIC_URL` | có | Worker tải video và upload keyframe qua R2 |
| `GROQ_API_KEY` | không | Chỉ dùng khi bật `ENABLE_LLM_SUMMARIZER`; notebook mặc định tắt và dùng LED (đầu ra English) |
| `OPENROUTER_API_KEY`, `HF_TOKEN` | không | Fallback LLM, pyannote |
| `GITHUB_TOKEN` | nếu repo private | Token chỉ có quyền đọc repo |
| `ENABLE_PARALLEL_BRANCHES`, `DENOISE_DEVICE`, `ENABLE_LLM_SUMMARIZER`, `ASR_ENGLISH_BACKEND`, `QWEN_ASR_BATCH_SIZE`, `KEYFRAME_CANDIDATES`, `ENABLE_OBJECT_TRACKING` | không | Công tắc tùy chọn: nếu tạo secret cùng tên (và bật quyền truy cập) thì giá trị đó được dùng; không thì dùng mặc định `true`, `cuda`, `false`, `qwen3`, `4`, `3`, `false` |
| `SCENE_SEGMENTER` | không | Cách chia scene: `content` (mặc định, PySceneDetect trên cả khung hình) hoặc `slide` (chỉ so vùng slide tĩnh, bỏ qua ô người nói; tự quay về `content` khi không xác định được vùng slide) |
| `SLIDE_ELEMENT_DETECTOR_WEIGHTS` | không | Đường dẫn checkpoint YOLO11n fine-tune (bảng, biểu đồ, ảnh, hình minh họa). Mặc định là bản seed 1 trên Drive (`DetTrack/lvvo-ft-261115-v1/yolo/yolo11n_ft_seed1/weights/selected.pt`); đặt `off` để tắt. Không thấy file thì bước này tự bỏ qua |

Runtime: **Runtime → Change runtime type → GPU** (T4 trở lên). Hướng dẫn đầy đủ: `docs/colab-gpu-worker-guide.md`.

In [24]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [25]:
# Cache model trên Drive để session sau không tải lại (~3 GB). Bỏ qua cell này nếu không cần.
from google.colab import drive
drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/mls-cache'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from google.colab import userdata

REPO_URL = 'https://github.com/multimodal-lecture-summarizer/multimodal-lecture-summarizer.git'
BRANCH = 'main'
REPO_DIR = '/content/mls'
PYTHON_BIN = '/content/venv/bin/python'

try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = REPO_URL.replace('https://', f'https://x-access-token:{token}@') if token else REPO_URL


def sh(cmd, **kwargs):
    subprocess.run(cmd, check=True, **kwargs)


def install_git_lfs():
    sh('apt-get update -qq && apt-get install -y -qq git-lfs', shell=True)


def clone_repo():
    # LFS files are fetched in the next phase, once git-lfs is installed.
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    if not os.path.isdir(REPO_DIR):
        sh(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO_DIR], env=env)
    else:
        # The runtime outlived an earlier run: move the checkout to the latest commit of BRANCH.
        sh(['git', 'fetch', '--depth', '1', 'origin', BRANCH], cwd=REPO_DIR, env=env)
        sh(['git', 'reset', '--hard', 'FETCH_HEAD'], cwd=REPO_DIR, env=env)
    sh(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)  # shows which commit this worker runs


def create_venv():
    if not Path(PYTHON_BIN).is_file():
        sh(['uv', 'venv', '--python', '3.11', '/content/venv'])


def pull_lfs():
    # Florence-2 weights live in Git LFS; the worker refuses to start without them.
    sh('git lfs install && git lfs pull', shell=True, cwd=REPO_DIR)


def install_requirements():
    sh(['uv', 'pip', 'install', '--python', PYTHON_BIN,
        '-r', 'ai_workers/requirements.txt', '-r', 'backend/requirements.txt',
        'setuptools'], cwd=REPO_DIR)  # PaddleOCR imports pkg_resources; uv venvs ship without setuptools
    # trackers declares numpy>=2 but runs on the pinned numpy 1.26.4, so it is installed without its deps.
    sh(['uv', 'pip', 'install', '--python', PYTHON_BIN, '--no-deps',
        '-r', 'ai_workers/requirements-tracking.txt'], cwd=REPO_DIR)


def run_parallel(*steps):
    with ThreadPoolExecutor(max_workers=len(steps)) as pool:
        for future in [pool.submit(step) for step in steps]:
            future.result()  # re-raises the first failure


# apt, clone and venv do not depend on each other; LFS needs apt+clone, install needs clone+venv.
run_parallel(install_git_lfs, clone_repo, create_venv)
run_parallel(pull_lfs, install_requirements)

sh([PYTHON_BIN, '-m', 'celery', '--version'])
print('Cài đặt hoàn tất.')

In [ ]:
import os
from google.colab import userdata

def secret(name, required=True):
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise RuntimeError(f'Thiếu Colab Secret: {name}')
        return ''

def setting(name, default):
    """Công tắc không nhạy cảm: dùng Colab Secret cùng tên nếu có, không thì dùng mặc định."""
    return secret(name, required=False).strip() or default

redis_url = secret('SHARED_REDIS_URL')
cache_dir = globals().get('CACHE_DIR', '/content/mls/cache')
site = '/content/venv/lib/python3.11/site-packages'

# YOLO11n fine-tune trên LVVO (bảng, biểu đồ, ảnh, hình minh họa): checkpoint seed 1 trên Drive.
# Không thấy file (chưa mount Drive, hoặc đặt secret = 'off') thì bỏ qua bước này.
SLIDE_WEIGHTS_DEFAULT = '/content/drive/MyDrive/DetTrack/lvvo-ft-261115-v1/yolo/yolo11n_ft_seed1/weights/selected.pt'
slide_weights = setting('SLIDE_ELEMENT_DETECTOR_WEIGHTS', SLIDE_WEIGHTS_DEFAULT)
if slide_weights.lower() == 'off' or not os.path.isfile(slide_weights):
    if slide_weights.lower() != 'off':
        print('Không thấy checkpoint phát hiện phần tử slide, bỏ qua bước này:', slide_weights)
    slide_weights = ''

worker_env = {
    **os.environ,
    'CELERY_BROKER_URL': redis_url,
    'CELERY_RESULT_BACKEND': redis_url,
    'CF_R2_ACCOUNT_ID': secret('CF_R2_ACCOUNT_ID'),
    'CF_R2_ACCESS_KEY_ID': secret('CF_R2_ACCESS_KEY_ID'),
    'CF_R2_SECRET_ACCESS_KEY': secret('CF_R2_SECRET_ACCESS_KEY'),
    'CF_R2_BUCKET_NAME': secret('CF_R2_BUCKET_NAME'),
    'CF_R2_PUBLIC_URL': secret('CF_R2_PUBLIC_URL'),
    'GROQ_API_KEY': secret('GROQ_API_KEY', required=False),
    'OPENROUTER_API_KEY': secret('OPENROUTER_API_KEY', required=False),
    'HF_TOKEN': secret('HF_TOKEN', required=False),
    'CACHE_DIR': cache_dir,
    'HF_HOME': f'{cache_dir}/huggingface',
    'FLORENCE_DEVICE': 'cuda',
    'SEMANTIC_CLIP_DEVICE': 'cuda',
    'PADDLEOCR_USE_GPU': '0',
    # Audio (ASR) and visual (scenes + Florence/CLIP) branches of a job run at the same time;
    # a branch that hits OOM is re-run sequentially. Set to 'false' to restore sequential runs.
    'ENABLE_PARALLEL_BRANCHES': setting('ENABLE_PARALLEL_BRANCHES', 'true'),
    'ENABLE_OBJECT_TRACKING': setting('ENABLE_OBJECT_TRACKING', 'false'),
    'SLIDE_ELEMENT_DETECTOR_WEIGHTS': slide_weights,
    # Cách chia scene: 'content' (PySceneDetect trên cả khung hình) hoặc 'slide' (chỉ so vùng slide
    # tĩnh, bỏ qua ô người nói; tự quay về 'content' khi không xác định được vùng slide).
    'SCENE_SEGMENTER': setting('SCENE_SEGMENTER', 'content'),
    # Ultralytics ghi cấu hình vào thư mục này; tránh ~/.config trên Colab.
    'YOLO_CONFIG_DIR': f'{cache_dir}/ultralytics',
    # DeepFilterNet on GPU (Colab has only 2 vCPU) and 4 audio chunks per Qwen3-ASR pass.
    'DENOISE_DEVICE': setting('DENOISE_DEVICE', 'cuda'),
    # English summaries come from the LED model; never call OpenRouter/Groq.
    'ENABLE_LLM_SUMMARIZER': setting('ENABLE_LLM_SUMMARIZER', 'false'),
    # 'qwen3' (chính xác hơn, nạp ~2 phút) hoặc 'whisper' (nhanh hơn).
    'ASR_ENGLISH_BACKEND': setting('ASR_ENGLISH_BACKEND', 'qwen3'),
    'QWEN_ASR_BATCH_SIZE': setting('QWEN_ASR_BATCH_SIZE', '4'),
    # Khung ứng viên mỗi scene khi chọn keyframe nét nhất; mỗi khung là một lần tua video (7 = như cũ).
    'KEYFRAME_CANDIDATES': setting('KEYFRAME_CANDIDATES', '3'),
    'CUBLAS_WORKSPACE_CONFIG': ':4096:8',
    'PROCESS_MIN_AVAILABLE_MEMORY_MB': '4096',
    'PROCESS_SOFT_MIN_AVAILABLE_MEMORY_MB': '3072',
    # Faster-Whisper loads cuDNN 9 / cuBLAS from the torch wheels.
    'LD_LIBRARY_PATH': f'{site}/nvidia/cudnn/lib:{site}/nvidia/cublas/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'PATH': '/content/venv/bin:' + os.environ['PATH'],
}
os.makedirs(cache_dir, exist_ok=True)
switches = ('ENABLE_PARALLEL_BRANCHES', 'ENABLE_OBJECT_TRACKING', 'SLIDE_ELEMENT_DETECTOR_WEIGHTS', 'SCENE_SEGMENTER',
            'DENOISE_DEVICE', 'ENABLE_LLM_SUMMARIZER', 'ASR_ENGLISH_BACKEND', 'QWEN_ASR_BATCH_SIZE',
            'KEYFRAME_CANDIDATES')
print('Cấu hình xong. Cache:', cache_dir)
print('Công tắc:', {k: worker_env[k] for k in switches})

In [30]:
from urllib.parse import urlsplit

url = worker_env["CELERY_BROKER_URL"]
parts = urlsplit(url)

assert url == url.strip(), "URL có khoảng trắng ở đầu hoặc cuối"
assert url.count("://") == 1, "URL bị dán lặp hoặc sai định dạng"
assert parts.scheme in ("redis", "rediss"), "Sai giao thức"
assert parts.hostname, "Thiếu hostname"
assert parts.port == 18916, "Sai port"
assert parts.path == "/0", "Database phải là /0"
assert worker_env["CELERY_RESULT_BACKEND"] == url, "Hai URL chưa giống nhau"

print("Định dạng Redis URL hợp lệ; chưa kiểm tra kết nối.")

Định dạng Redis URL hợp lệ; chưa kiểm tra kết nối.


In [ ]:
import csv, os, re, subprocess, time
from datetime import datetime

python_bin = "/content/venv/bin/python"

# Nếu Celery chưa được cài, dừng ngay tại đây.
subprocess.run(
    [python_bin, "-m", "celery", "--version"],
    check=True,
)

# Thời gian xử lý mỗi video. Worker in một dòng [TIMING] khi xong job (tính từ sau khi tải video);
# notebook đo thêm thời gian thực từ lúc bắt đầu tải video, rồi ghi một dòng vào job_timings.csv.
TIMINGS_CSV = os.path.join(worker_env["CACHE_DIR"], "job_timings.csv")
JOB_START = re.compile(r"Downloading R2 video securely using boto3 from: .*/([^/\s]+)\.mp4")
TIMING = re.compile(r"\[TIMING\] total=([\d.]+)s, video=([\d.]+)s(?:, realtime_factor=([\d.]+)x)? \| stage spans: (.*)")
TIMING_FIELDS = ["finished_at", "video_id", "video_seconds", "pipeline_seconds", "wall_seconds",
                 "realtime_factor", "audio_s", "visual_s", "semantic_s", "timeline_s", "summarizing_s", "stage_spans"]
job = {}


def record_timing(match):
    total, video, rtf, spans_text = match.groups()
    spans = dict(re.findall(r"(\w+)=([\d.]+)s", spans_text))
    wall = time.monotonic() - job["started"] if "started" in job else None
    row = {
        "finished_at": datetime.now().isoformat(timespec="seconds"),
        "video_id": job.get("video_id", ""),
        "video_seconds": video,
        "pipeline_seconds": total,
        "wall_seconds": f"{wall:.1f}" if wall is not None else "",
        "realtime_factor": rtf or "",
        "audio_s": spans.get("audio", ""),
        "visual_s": spans.get("visual", ""),
        "semantic_s": spans.get("semantic", ""),
        "timeline_s": spans.get("timeline", ""),
        "summarizing_s": spans.get("summarizing", ""),
        "stage_spans": spans_text.strip(),
    }
    new_file = not os.path.exists(TIMINGS_CSV)
    with open(TIMINGS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=TIMING_FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)
    video_min = float(video) / 60
    print(
        f"\n=== Thời gian xử lý video {row['video_id'] or '(không rõ id)'} ===\n"
        f"Độ dài video: {video_min:.1f} phút ({video}s)\n"
        f"Pipeline (sau khi tải video): {total}s"
        + (f" · {rtf}x thời lượng video" if rtf else "") + "\n"
        + (f"Thời gian thực, gồm tải video: {row['wall_seconds']}s\n" if row["wall_seconds"] else "")
        + f"Các chặng: audio {row['audio_s']}s · visual {row['visual_s']}s · semantic {row['semantic_s']}s"
        f" · tóm tắt {row['summarizing_s']}s\n"
        f"Đã ghi vào {TIMINGS_CSV}\n",
        flush=True,
    )
    job.clear()


proc = subprocess.Popen(
    [
        python_bin, "-m", "celery",
        "-A", "ai_workers.core.celery_app",
        "worker",
        "--loglevel=info",
        "--pool=solo",
        "--concurrency=1",
        "-Q", "gpu",
        "-n", "colab@%h",
    ],
    cwd="/content/mls",
    env=worker_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

try:
    for line in proc.stdout:
        print(line, end="", flush=True)
        started = JOB_START.search(line)
        if started:
            job.update(started=time.monotonic(), video_id=started.group(1))
            continue
        timing = TIMING.search(line)
        if timing:
            record_timing(timing)

    print("\nWorker kết thúc, mã thoát:", proc.wait())

except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print("Worker đã dừng.")